In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# Check if the current `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
# is available, and if not, use the CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
def compare_tensor_memory():
    # 1. Create a dense weight matrix (e.g., a 1000x1000 Linear layer)
    # Total elements: 1,000,000
    dense_tensor = torch.randn(1000, 1000, dtype=torch.float32)
    
    # 2. Simulate 90% Unstructured Pruning (Zeroing out 90% of weights)
    mask = torch.rand(1000, 1000) > 0.90
    dense_pruned_tensor = dense_tensor * mask
    
    # 3. Convert the pruned dense tensor to a Sparse Tensor (COO format)
    sparse_tensor = dense_pruned_tensor.to_sparse()
    
    # 4. Calculate Memory of Dense Tensor
    # float32 = 4 bytes per element
    dense_bytes = dense_pruned_tensor.numel() * 4 
    
    # 5. Calculate Memory of Sparse Tensor
    # Values: float32 (4 bytes). Indices: int64 (8 bytes per dimension)
    nnz = sparse_tensor._nnz() # Number of non-zero elements
    sparse_values_bytes = nnz * 4
    sparse_indices_bytes = nnz * sparse_tensor.dim() * 8
    sparse_total_bytes = sparse_values_bytes + sparse_indices_bytes
    
    # 6. Output the comparison
    print(f"--- Memory Comparison (90% Sparsity) ---")
    print(f"Dense Tensor Memory:  {dense_bytes / 1024 / 1024:.2f} MB")
    print(f"Sparse Tensor Memory: {sparse_total_bytes / 1024 / 1024:.2f} MB")
    print(f"Memory Saved:         {(dense_bytes - sparse_total_bytes) / 1024 / 1024:.2f} MB")
    
    return dense_pruned_tensor, sparse_tensor

# Execute the function
dense_t, sparse_t = compare_tensor_memory()

In [7]:
# Below we are preprocessing data for CIFAR-10. We use an arbitrary batch size of 128.
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Loading the CIFAR-10 dataset:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

100%|██████████| 170M/170M [00:54<00:00, 3.11MB/s] 
/home/prasanna/coding/transformers-playground/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [8]:
#Dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [9]:
# Deeper neural network class to be used as teacher:
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Lightweight neural network class to be used as student:
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super(LightNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [10]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            # inputs: A collection of batch_size images
            # labels: A vector of dimensionality batch_size with integers denoting class of each image
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            # outputs: Output of the network for the collection of images. A tensor of dimensionality batch_size x num_classes
            # labels: The actual labels of the images. Vector of dimensionality batch_size
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [11]:
torch.manual_seed(42)
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)

# Instantiate the lightweight network:
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)

Epoch 1/10, Loss: 1.344755276389744
Epoch 2/10, Loss: 0.8834929441856911
Epoch 3/10, Loss: 0.689524745239931
Epoch 4/10, Loss: 0.5526437603146829
Epoch 5/10, Loss: 0.4360969790336116
Epoch 6/10, Loss: 0.3277716593592978
Epoch 7/10, Loss: 0.22849296469746344
Epoch 8/10, Loss: 0.18547722767762212
Epoch 9/10, Loss: 0.14644855271329354
Epoch 10/10, Loss: 0.12917063433838927
Test Accuracy: 74.96%


In [13]:
torch.manual_seed(42)
new_nn_light = LightNN(num_classes=10).to(device)

# Print the norm of the first layer of the initial lightweight model
print("Norm of 1st layer of nn_light:", torch.norm(nn_light.features[0].weight).item())
# Print the norm of the first layer of the new lightweight model
print("Norm of 1st layer of new_nn_light:", torch.norm(new_nn_light.features[0].weight).item())

Norm of 1st layer of nn_light: 2.327361822128296
Norm of 1st layer of new_nn_light: 2.327361822128296


In [14]:
total_params_deep = "{:,}".format(sum(p.numel() for p in nn_deep.parameters()))
print(f"DeepNN parameters: {total_params_deep}")
total_params_light = "{:,}".format(sum(p.numel() for p in nn_light.parameters()))
print(f"LightNN parameters: {total_params_light}")

DeepNN parameters: 1,186,986
LightNN parameters: 267,738


In [15]:
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light_ce = test(nn_light, test_loader, device)

Epoch 1/10, Loss: 1.4696883405261028
Epoch 2/10, Loss: 1.1583035775767567
Epoch 3/10, Loss: 1.0218140984435216
Epoch 4/10, Loss: 0.9155943631516088
Epoch 5/10, Loss: 0.8383376859033199
Epoch 6/10, Loss: 0.7742130558204163
Epoch 7/10, Loss: 0.7088720432632719
Epoch 8/10, Loss: 0.6535516749410069
Epoch 9/10, Loss: 0.6002736132772987
Epoch 10/10, Loss: 0.5533886973357871
Test Accuracy: 70.68%


In [16]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light_ce:.2f}%")

Teacher accuracy: 74.96%
Student accuracy: 70.68%


so far the training of the student & teacher is done now we're about to being the knowledge distillation with the help of teacher improving the student accuracy 


we trained the 1st student with accrauxy of 70%
now we train the another student from scratch using the knowledge of teacher 

In [ ]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # Forward pass with the student model
            student_logits = student(inputs)

            #Soften the student logits by applying softmax first and log() second
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)


            # we can also use KLDiv from torch which does exact same this 
            # soft_targets_loss = nn.functional.kl_div(soft_prob,soft_targets) * (T**2)
            # -----------------------
            # Calculate the soft targets loss. Scaled by T**2 as suggested by the authors of the paper "Distilling the knowledge in a neural network"
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")



Epoch 1/10, Loss: 2.386768663935649
Epoch 2/10, Loss: 1.8655977346708097
Epoch 3/10, Loss: 1.6386952278254283
Epoch 4/10, Loss: 1.4777717075079604
Epoch 5/10, Loss: 1.349039863137638
Epoch 6/10, Loss: 1.2313818399558591
Epoch 7/10, Loss: 1.141650965299143
Epoch 8/10, Loss: 1.0517110513604206
Epoch 9/10, Loss: 0.9809168580243045
Epoch 10/10, Loss: 0.9094337410939014
Test Accuracy: 70.90%
Teacher accuracy: 74.96%
Student accuracy without teacher: 70.68%
Student accuracy with CE + KD: 70.90%


In [26]:
# Apply ``train_knowledge_distillation`` with a temperature of 2. Arbitrarily set the weights to 0.75 for CE and 0.25 for distillation loss.
train_knowledge_distillation(
    teacher=nn_deep,
    student=new_nn_light,
    train_loader=train_loader,
    epochs=10,
    learning_rate=0.001,
    T=3,
    soft_target_loss_weight=0.25,
    ce_loss_weight=0.75,
    device=device
 )
test_accuracy_light_ce_and_kd = test(new_nn_light, test_loader, device)

# Compare the student test accuracy with and without the teacher, after distillation
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

Epoch 1/10, Loss: 1.2389017398400075
Epoch 2/10, Loss: 1.122143330476473
Epoch 3/10, Loss: 1.0531044770079805
Epoch 4/10, Loss: 0.9857105067014085
Epoch 5/10, Loss: 0.9302403408548107
Epoch 6/10, Loss: 0.8783187491204733
Epoch 7/10, Loss: 0.8370403545286954
Epoch 8/10, Loss: 0.7908896090429457
Epoch 9/10, Loss: 0.7497511457299333
Epoch 10/10, Loss: 0.7130641065290212
Test Accuracy: 71.80%
Teacher accuracy: 74.96%
Student accuracy without teacher: 70.68%
Student accuracy with CE + KD: 71.80%


In [28]:
d = torch.nn.functional.softmax(torch.tensor([2.0,1.0,0.0]), dim=-1)
d

tensor([0.6652, 0.2447, 0.0900])